In [6]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

In [3]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd
import numpy as np
import time
import json
from datetime import datetime, timedelta
from tqdm import tqdm

load_dotenv() 

True

# 1. 농수축산 도매시장 코드 추출

In [ ]:
# 농수축산 도매시장 코드
API_KEY = os.getenv("API_KEY")
START_INDEX = 1
END_INDEX=1000


data_list = []

while START_INDEX<=17001:
    url = f"http://211.237.50.150:7080/openapi/{API_KEY}/json/Grid_20240626000000000668_1/{START_INDEX}/{END_INDEX}"
    # API 호출
    response = requests.get(url)

    # 결과 확인 및 데이터 추가
    if response.status_code == 200:
        json_data = response.json()
        data = json_data.get('Grid_20240626000000000668_1')
        data_list.extend(data.get('row'))
        print(f"API 호출 작업중 : {START_INDEX} - {END_INDEX}")
    else:
        print(f"API 호출 실패 - 코드 : {response.status_code}, 작업중 : {START_INDEX} - {END_INDEX}" )
        break

    # 다음 사이클로 이동
    START_INDEX += 1000
    if START_INDEX==17001:
        END_INDEX = 17555
    else:
        END_INDEX += 1000
    time.sleep(0.5)

df = pd.DataFrame(data_list)
df.to_csv(f'농축산_품목코드.csv', encoding='cp949')

# 2. 경락데이터 실시간 가져오기(24년 5월 2일~)

In [13]:
# 경락데이터 가져오기
# https://data.mafra.go.kr/opendata/data/indexOpenDataDetail.do?data_id=20240625000000002460

API_KEY = os.getenv("API_KEY")
API_URL='Grid_20240625000000000654_1'
df_market = pd.read_csv('도매시장_코드.csv', encoding='cp949')

ITEM_CODES = {
    "양파": "1201",
    "배추": "1001",
    "상추": "1005",
    "사과": "0601",
    "쌀": "0103"
}

max_retries = 3
FAIL_LOG = []

# 아이템 코드 별 반복 시작
for item, icode in tqdm(ITEM_CODES.items(), desc="전체 품목 진행"):  # 품목 진행률
    LARGE = icode[:2]
    MID = icode[2:]
    data_list = []
    
    
    #날짜 설정 (수기 설정)
    start_date = '20150101'
    end_date = '20150115'
    date_str= start_date
    
    start = datetime.strptime(start_date, '%Y%m%d')
    end = datetime.strptime(end_date, '%Y%m%d')
    total_days = (end-start).days
    cnt=1
    progress=0
    print(f'{item} 작업 시작 - 진척 현황 => ')
    print('[ ', end='')
    
    #날짜에 대한 반복문 실시 / 기본 인덱스 1천개씩 
    while date_str <= end_date:
        START_INDEX = 1
        END_INDEX=1000
        

        # 마켓 별 반복문
        for mcode, market in df_market.values:
            retry_count = 0
            market_success = False  # 마켓별 성공 플래그
            while retry_count < max_retries :
                # API 호출
                url = f"http://211.237.50.150:7080/openapi/{API_KEY}/json/{API_URL}/{START_INDEX}/{END_INDEX}?SALEDATE={date_str}&WHSALCD={mcode}&LARGE={LARGE}&MID={MID}"
                

                try:
                    response = requests.get(url, timeout=5)

                    # 결과 확인 및 데이터 추가
                    if response.status_code == 200:
                        json_data = response.json()
                        data = json_data.get(API_URL)
                        rows = data.get('row', [])
                        total = data.get('totalCnt', 0)
                        print(rows)
                        if rows:
                            data_list.extend(rows)
                            market_success = True

                        # 다음 페이지로 이동
                        if total > END_INDEX:
                            START_INDEX += 1000
                            END_INDEX += 1000
                        else:
                            time.sleep(0.1)
                            #print('=', end='')  # 쌓이는지 체크
                            break

                    else:
                        print(f"API 호출 실패 - 코드 : {response.status_code}, 작업중 : {START_INDEX} - {END_INDEX}" )
                        retry_count += 1
                        break

                except requests.exceptions.ConnectTimeout:
                    print(f"[{item}] 타임아웃 발생 ({retry_count+1}/{max_retries}) {url}")
                    time.sleep(2)
                except requests.exceptions.RequestException as e:
                    print(f"[{item}] 기타 요청 예외 발생: {e} ({retry_count+1}/{max_retries})")
                    time.sleep(2)
            
                if not market_success:
                    # 실패 로그 기록
                    FAIL_LOG.append({
                        "item": item,
                        "mcode": mcode,
                        "market": market,
                        "date": date_str,
                        "url": url
                    })
                    # 현재까지 저장
                    if data_list:
                        df = pd.DataFrame(data_list)
                        df.to_csv(f'data/도매시장_실시간_경락_정보_{item}_{start_date}-{date_str}_Fail.csv', encoding='cp949', index=False)
                        print(f"[{item}] {date_str} 실패로 조기 저장됨")
                        break

                time.sleep(0.5)
        
        
        # 하루씩 날짜 더함
        type_date = datetime.strptime(date_str, '%Y%m%d')
        current_date = type_date + timedelta(days=1)
        date_str = current_date.strftime('%Y%m%d')
        cnt += 1
        time.sleep(1)
        print('=', end='')
#         if cnt%(total_days//100)==0:
#             progress += 1
#             print(f'{progress}%', end='')
        
    print(']')    
    # 데이터프레임 저장    
    if data_list:
        df = pd.DataFrame(data_list)
        df.to_csv(f'data/도매시장_실시간_경락_정보_{item}_{start_date}-{end_date}.csv', encoding='cp949')
        print(f'<--{item}_{start_date}-{end_date}_{df.shape[0]}행 작업 완료-->')
        print()
        time.sleep(3)
    else:
        print(f'{item}_{start_date}-{end_date}_데이터 없음')

# 실패 로그 저장
if FAIL_LOG:
    df_fail = pd.DataFrame(FAIL_LOG)
    df_fail.to_csv('data/fail_log.csv', index=False, encoding='cp949')
    print(f"❗ 실패 요청 {len(FAIL_LOG)}건 기록됨: data/fail_log.csv")    


전체 품목 진행:   0%|                                                                            | 0/5 [00:00<?, ?it/s]

양파 작업 시작 - 진척 현황 => 
[ []
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]

전체 품목 진행:  20%|█████████████▍                                                     | 1/5 [01:59<07:57, 119.31s/it]

=]
양파_20150101-20150115_데이터 없음
배추 작업 시작 - 진척 현황 => 
[ []
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]


전체 품목 진행:  20%|█████████████▍                                                     | 1/5 [02:15<09:00, 135.21s/it]

=

KeyboardInterrupt: 

# 3. 경락데이터 요약 데이터
- 전국 도매시장 일별 경락가격 (2013-01-02 ~ 2023-12-31)
- https://data.mafra.go.kr/opendata/data/indexOpenDataDetail.do?data_id=20141216000000000367
* 너무 느려서 탈락 -> https://data.mafra.go.kr/opendata/data/indexOpenDataDetail.do?data_id=20141014000000000089

In [11]:
import os
import requests
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm
import time
from dotenv import load_dotenv

load_dotenv() 
# API 키 및 URL 설정
# API_KEY는 환경 변수에서 가져옵니다.
# 실제 실행 시에는 유효한 API_KEY를 환경 변수에 설정해야 합니다.
API_KEY = os.getenv("API_KEY")
if not API_KEY:
    print("API_KEY 환경 변수가 설정되지 않았습니다. API 키를 설정해주세요.")
    # 샘플 검증 및 코드 실행을 위해 임시 플레이스홀더를 사용합니다.
    # 실제 API 호출 시에는 이 값을 유효한 API 키로 대체해야 합니다.
    API_KEY = "YOUR_API_KEY_HERE" 

# 새로운 API URL 설정
API_URL = 'Grid_20150401000000000216_1'  # Grid_20141119000000000012_1 너무 느림

# 품목코드
ITEM_CODES = {
    "양파": "1201",
    "배추": "1001",
    "상추": "1005",
    "사과": "0601",
    "쌀": "0103"
}
ITEM_CODE = "1201"

max_retries = 3 # API 호출 재시도 횟수
FAIL_LOG = [] # 실패한 요청을 기록할 리스트

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True) # 데이터를 저장할 'data' 디렉토리 생성

# 날짜 설정
# 예시를 위해 짧은 기간으로 설정했습니다. 원하는 기간으로 변경하세요.
start_date_str = '20230101'
end_date_str = '20230103' 

start_dt = datetime.strptime(start_date_str, '%Y%m%d')
end_dt = datetime.strptime(end_date_str, '%Y%m%d')

current_dt = start_dt
all_data_list = [] # 전체 데이터를 저장할 리스트

print(f'{start_date_str}부터 {end_date_str}까지 농수산물 가격/거래량 데이터 수집 시작')

# 날짜별 데이터 수집을 위한 반복문
# tqdm을 사용하여 전체 날짜 진행률을 표시합니다.
with tqdm(total=(end_dt - start_dt).days + 1, desc="전체 날짜 진행") as pbar_date:
    while current_dt <= end_dt:
        date_str = current_dt.strftime('%Y%m%d') # 현재 날짜 문자열
        
        START_INDEX = 1 # API 호출 시작 인덱스
        END_INDEX = 1000 # API 호출 종료 인덱스 (한 번에 가져올 데이터 수)
        date_data_list = [] # 현재 날짜의 데이터를 임시로 저장할 리스트
        date_success = False # 현재 날짜 데이터 수집 성공 여부 플래그

        print(f"\n[{date_str}] 작업 시작...")

        # 페이지네이션 처리를 위한 반복문
        while True:
            # API URL 구성: 시장 코드(WHSALCD), 대분류(LARGE), 중분류(MID)는 더 이상 필요 없음
            url = f"http://211.237.50.150:7080/openapi/{API_KEY}/json/{API_URL}/{START_INDEX}/{END_INDEX}?AUCNG_DE={date_str}&PRDLST_CD={ITEM_CODE}"
            retry_count = 0 # 현재 페이지에 대한 재시도 횟수
            
            # API 호출 재시도 루프
            while retry_count < max_retries:
                try:
                    response = requests.get(url, timeout=10) # API 호출, 타임아웃 10초 설정

                    if response.status_code == 200:
                        json_data = response.json()
                        data = json_data.get(API_URL) # API 응답에서 실제 데이터 부분 추출
                        
                        if data:
                            rows = data.get('row', []) # 실제 데이터 행 리스트
                            total = data.get('totalCnt', 0) # 전체 데이터 수
                            print('totalCnt :', total)
                            
                            if rows: # 데이터가 존재하면 리스트에 추가
                                date_data_list.extend(rows)
                                date_success = True # 현재 날짜 데이터 수집 성공으로 간주
                                print('=', end='')
                                time.sleep(0.1) # 서버 부하를 줄이기 위한 짧은 지연
                            
                            # 다음 페이지로 이동하거나 현재 날짜 데이터 수집 완료
                            if total > END_INDEX: # 전체 데이터가 현재 페이지 범위보다 크면 다음 페이지 요청
                                START_INDEX += 1000
                                END_INDEX += 1000
                                time.sleep(0.1) # 서버 부하를 줄이기 위한 짧은 지연
                            else: # 현재 날짜의 모든 데이터 수집 완료
                                print(f'{date_str} 리스트 추가 완료')
                                break 
                        else:
                            print(f"[{date_str}] API 응답에 '{API_URL}' 키가 없습니다. URL: {url}")
                            retry_count += 1
                            time.sleep(2) # 재시도 전 대기
                            continue # 다음 재시도
                        break # 성공적으로 데이터 처리 후 현재 페이지 재시도 루프 탈출
                    else:
                        print(f"API 호출 실패 - 코드: {response.status_code}, URL: {url}")
                        retry_count += 1
                        time.sleep(2) # 실패 시 재시도 전 대기
                
                except requests.exceptions.ConnectTimeout:
                    print(f"[{date_str}] 타임아웃 발생 ({retry_count+1}/{max_retries}) URL: {url}")
                    retry_count += 1
                    time.sleep(2)
                except requests.exceptions.RequestException as e:
                    print(f"[{date_str}] 기타 요청 예외 발생: {e} ({retry_count+1}/{max_retries}) URL: {url}")
                    retry_count += 1
                    time.sleep(2)
                except Exception as e: # 예상치 못한 모든 오류 처리
                    print(f"[{date_str}] 알 수 없는 오류 발생: {e} ({retry_count+1}/{max_retries}) URL: {url}")
                    retry_count += 1
                    time.sleep(2)

            if retry_count == max_retries: # 모든 재시도 실패 시
                print(f"[{date_str}] {max_retries}번 재시도 후에도 API 호출 실패. 다음 날짜로 이동합니다.")
                FAIL_LOG.append({
                    "date": date_str,
                    "url": url,
                    "error": "Max retries reached or API response error"
                })
                break # 현재 날짜 처리 중단, 다음 날짜로 이동

            if not date_success and START_INDEX == 1: # 첫 페이지에서 데이터가 없으면 (즉, 해당 날짜에 데이터가 없음)
                print(f"[{date_str}] 해당 날짜에 데이터가 없거나 첫 페이지 호출에 실패했습니다.")
                break # 현재 날짜 처리 중단, 다음 날짜로 이동
            
            if total <= END_INDEX and START_INDEX > 1: # 마지막 페이지까지 데이터 수집 완료 (total이 END_INDEX보다 작거나 같고, 이미 첫 페이지가 아닌 경우)
                break

        if date_data_list: # 현재 날짜에 수집된 데이터가 있으면 전체 리스트에 추가
            all_data_list.extend(date_data_list)
            print(f"[{date_str}] {len(date_data_list)}건의 데이터 수집 완료.")
        else:
            print(f"[{date_str}] 수집된 데이터가 없습니다.")

        current_dt += timedelta(days=1) # 다음 날짜로 이동
        pbar_date.update(1) # tqdm 진행률 업데이트
        time.sleep(0.5) # 날짜 변경 전 짧은 대기

# 전체 데이터프레임으로 변환 및 CSV 저장
if all_data_list:
    df = pd.DataFrame(all_data_list)
    output_filename = os.path.join(DATA_DIR, f'농수산물_가격_거래량_{start_date_str}-{end_date_str}.csv')
    df.to_csv(output_filename, encoding='cp949', index=False)
    print(f"\n<-- 전체 데이터 {df.shape[0]}행 작업 완료: {output_filename} -->")
else:
    print("\n수집된 전체 데이터가 없습니다.")

# 실패 로그 저장
if FAIL_LOG:
    df_fail = pd.DataFrame(FAIL_LOG)
    fail_log_filename = os.path.join(DATA_DIR, 'fail_log.csv')
    df_fail.to_csv(fail_log_filename, index=False, encoding='cp949')
    print(f"❗ 실패 요청 {len(FAIL_LOG)}건 기록됨: {fail_log_filename}")

print("\n--- 샘플 URL 및 예상 응답 구조 검증 ---")
# 샘플 URL 생성 (실제 API 키 필요)
sample_date = '20230101'
sample_start_index = 1
sample_end_index = 10
sample_url = f"http://211.237.50.150:7080/openapi/{API_KEY}/json/{API_URL}/{sample_start_index}/{sample_end_index}?AUCNG_DE={sample_date}"
print(f"생성된 샘플 URL: {sample_url}")

# 예상되는 JSON 응답 구조 (실제 API 호출 없이 구조만 시뮬레이션)
print("\n이 API 호출에 대한 예상 JSON 응답 구조:")
print(f"""
{{
    "{API_URL}": {{
        "totalCnt": 12345, # 전체 데이터 건수 (이 값이 1000개를 넘으면 페이지네이션이 작동합니다)
        "row": [
            {{
                "AUCNG_DE": "{sample_date}", # 경매 일자
                "PUMM_NM": "사과", # 품목명
                "PRDLFC_GU_NM": "후지", # 품종 구분명
                "GRAD_NM": "특", # 등급명
                "LAUNC_AM": 10000, # 경락 금액
                "TRNS_QTY": 50, # 거래량
                "STND_UNIT_NM": "KG" # 규격 단위명
                // ... 기타 필드 (실제 API 응답에 따라 달라질 수 있습니다)
            }},
            {{
                // ... 다음 데이터 행
            }}
        ]
    }}
}}
""")
print("\n위 구조에서 'totalCnt' 값을 확인하여 1000개를 초과하는 경우, START_INDEX와 END_INDEX가 자동으로 증가하며 다음 페이지의 데이터를 요청합니다.")
print("이 코드를 실행하기 전에 `API_KEY` 환경 변수를 유효한 값으로 설정해야 실제 데이터를 성공적으로 가져올 수 있습니다.")


20230101부터 20230103까지 농수산물 가격/거래량 데이터 수집 시작


전체 날짜 진행:  33%|██████████████████████▋                                             | 1/3 [00:00<00:00,  7.55it/s]


[20230101] 작업 시작...
totalCnt : 0
20230101 리스트 추가 완료
[20230101] 해당 날짜에 데이터가 없거나 첫 페이지 호출에 실패했습니다.
[20230101] 수집된 데이터가 없습니다.

[20230102] 작업 시작...
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 완료
totalCnt : 8
=20230102 리스트 추가 

전체 날짜 진행:  33%|██████████████████████▋                                             | 1/3 [00:09<00:18,  9.21s/it]


KeyboardInterrupt: 

In [ ]:
import os
import requests
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm
import time
from dotenv import load_dotenv # dotenv 라이브러리 임포트

load_dotenv() # .env 파일에서 환경 변수 로드

# API 키 및 URL 설정
# API_KEY는 환경 변수에서 가져옵니다.
# 실제 실행 시에는 유효한 API_KEY를 환경 변수에 설정해야 합니다.
API_KEY = os.getenv("API_KEY")
if not API_KEY:
    print("API_KEY 환경 변수가 설정되지 않았습니다. API 키를 설정해주세요.")
    # 샘플 검증 및 코드 실행을 위해 임시 플레이스홀더를 사용합니다.
    # 실제 API 호출 시에는 이 값을 유효한 API 키로 대체해야 합니다.
    API_KEY = "YOUR_API_KEY_HERE" 

# 새로운 API URL 설정
API_URL = 'Grid_20150401000000000216_1' # 변경된 API URL



max_retries = 3 # API 호출 재시도 횟수
FAIL_LOG = [] # 실패한 요청을 기록할 리스트

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True) # 데이터를 저장할 'data' 디렉토리 생성

# 품목코드
ITEM_CODES = {
    "양파": "1201",
    "배추": "1001",
    "상추": "1005",
    "사과": "0601",
    "쌀": "0103"
}

for item, code in tqdm(ITEM_CODES.items(), desc="전체 품목 진행"):  # 품목 진행률
        
    # 날짜 설정
    # 예시를 위해 짧은 기간으로 설정했습니다. 원하는 기간으로 변경하세요.
    start_date_str = '20200101'
    end_date_str = '20231231' 

    start_dt = datetime.strptime(start_date_str, '%Y%m%d')
    end_dt = datetime.strptime(end_date_str, '%Y%m%d')

    current_dt = start_dt
    all_data_list = [] # 전체 데이터를 저장할 리스트

    print(f'{start_date_str}부터 {end_date_str}까지 농수산물 가격/거래량 데이터 수집 시작 (품목: {item})')

    # 날짜별 데이터 수집을 위한 반복문
    # tqdm을 사용하여 전체 날짜 진행률을 표시합니다.
    with tqdm(total=(end_dt - start_dt).days + 1, desc="전체 날짜 진행") as pbar_date:
        while current_dt <= end_dt:
            date_str = current_dt.strftime('%Y%m%d') # 현재 날짜 문자열

            START_INDEX = 1 # API 호출 시작 인덱스
            END_INDEX = 1000 # API 호출 종료 인덱스 (한 번에 가져올 데이터 수)
            date_data_list = [] # 현재 날짜의 데이터를 임시로 저장할 리스트

            print(f"\n[{date_str}] 작업 시작...")

            # 현재 날짜에 대한 페이지네이션 루프를 제어하는 플래그
            done_for_this_date = False 

            # 페이지네이션 처리를 위한 반복문
            while not done_for_this_date:
                # API URL 구성: PRDLST_CD (품목코드) 추가
                url = f"http://211.237.50.150:7080/openapi/{API_KEY}/json/{API_URL}/{START_INDEX}/{END_INDEX}?AUCNG_DE={date_str}&PRDLST_CD={code}"
                retry_count = 0 # 현재 페이지에 대한 재시도 횟수

                # API 호출 재시도 루프
                while retry_count < max_retries:
                    try:
                        response = requests.get(url, timeout=10) # API 호출, 타임아웃 10초 설정

                        if response.status_code == 200:
                            json_data = response.json()
                            data = json_data.get(API_URL) # API 응답에서 실제 데이터 부분 추출

                            if data:
                                rows = data.get('row', []) # 실제 데이터 행 리스트
                                total = data.get('totalCnt', 0) # 전체 데이터 수
                                print(f'totalCnt for {date_str} (page {START_INDEX}-{END_INDEX}): {total}')

                                if rows: # 데이터가 존재하면 리스트에 추가
                                    date_data_list.extend(rows)
                                    print('=', end='')
                                    time.sleep(0.1) # 서버 부하를 줄이기 위한 짧은 지연

                                    # 다음 페이지로 이동하거나 현재 날짜 데이터 수집 완료
                                    if total > END_INDEX: # 전체 데이터가 현재 페이지 범위보다 크면 다음 페이지 요청
                                        START_INDEX += 1000
                                        END_INDEX += 1000
                                        break # 성공적으로 응답 받았으니 재시도 루프 탈출, 다음 페이지로 이동
                                    else: # 현재 날짜의 모든 데이터 수집 완료
                                        print(f'\n[{date_str}] 리스트 추가 완료 (총 {len(date_data_list)}건)')
                                        done_for_this_date = True # 이 날짜 작업 완료
                                        break # 재시도 루프 탈출
                                else: # 현재 페이지에 row 데이터가 없는 경우 (totalCnt가 0이거나, 더 이상 가져올 데이터가 없는 경우)
                                    print(f"\n[{date_str}] 현재 페이지에 데이터가 없으며, 더 이상 가져올 데이터가 없습니다.")
                                    done_for_this_date = True # 이 날짜 작업 완료
                                    break # 재시도 루프 탈출
                            else: # API 응답에 API_URL 키가 없는 경우 (예: 응답 구조 오류)
                                print(f"\n[{date_str}] API 응답에 '{API_URL}' 키가 없습니다. URL: {url}")
                                retry_count += 1
                                time.sleep(2) # 재시도 전 대기
                                continue # 다음 재시도
                        else: # 200이 아닌 응답 코드
                            print(f"\nAPI 호출 실패 - 코드: {response.status_code}, URL: {url}")
                            retry_count += 1
                            time.sleep(2)

                    except requests.exceptions.ConnectTimeout:
                        print(f"\n[{date_str}] 타임아웃 발생 ({retry_count+1}/{max_retries}) URL: {url}")
                        retry_count += 1
                        time.sleep(2)
                    except requests.exceptions.RequestException as e:
                        print(f"\n[{date_str}] 기타 요청 예외 발생: {e} ({retry_count+1}/{max_retries}) URL: {url}")
                        retry_count += 1
                        time.sleep(2)
                    except Exception as e: # 예상치 못한 모든 오류 처리
                        print(f"\n[{date_str}] 알 수 없는 오류 발생: {e} ({retry_count+1}/{max_retries}) URL: {url}")
                        retry_count += 1
                        time.sleep(2)

                # 재시도 루프를 모두 돌았는데도 done_for_this_date가 True가 되지 않았다면,
                # 모든 재시도가 실패한 것이므로 이 날짜에 대한 작업을 중단한다.
                if not done_for_this_date: # 즉, retry_count == max_retries 이고 성공적으로 break 하지 못한 경우
                    print(f"\n[{date_str}] {max_retries}번 재시도 후에도 API 호출 실패. 다음 날짜로 이동합니다.")
                    FAIL_LOG.append({
                        "date": date_str,
                        "url": url,
                        "error": "Max retries reached for a page"
                    })
                    done_for_this_date = True # 이 날짜 작업 완료 (실패로)

                # done_for_this_date가 True가 되면 while not done_for_this_date 루프가 종료된다.

            if date_data_list: # 현재 날짜에 수집된 데이터가 있으면 전체 리스트에 추가
                all_data_list.extend(date_data_list)
                print(f"[{date_str}] {len(date_data_list)}건의 데이터 최종 수집 완료.")
            else:
                print(f"[{date_str}] 수집된 데이터가 없습니다.")

            current_dt += timedelta(days=1) # 다음 날짜로 이동
            pbar_date.update(1) # tqdm 진행률 업데이트
            time.sleep(0.5) # 날짜 변경 전 짧은 대기

    # 전체 데이터프레임으로 변환 및 CSV 저장
    if all_data_list:
        df = pd.DataFrame(all_data_list)
        output_filename = os.path.join(DATA_DIR, f'농수산물_가격_거래량_{item}_{start_date_str}-{end_date_str}.csv')
        df.to_csv(output_filename, encoding='cp949', index=False)
        print(f"\n<-- 전체 데이터 {df.shape[0]}행 작업 완료: {output_filename} -->")
    else:
        print("\n수집된 전체 데이터가 없습니다.")

# 실패 로그 저장
if FAIL_LOG:
    df_fail = pd.DataFrame(FAIL_LOG)
    fail_log_filename = os.path.join(DATA_DIR, 'fail_log.csv')
    df_fail.to_csv(fail_log_filename, index=False, encoding='cp949')
    print(f"❗ 실패 요청 {len(FAIL_LOG)}건 기록됨: {fail_log_filename}")


전체 품목 진행:   0%|                                                                            | 0/5 [00:00<?, ?it/s]

20200101부터 20231231까지 농수산물 가격/거래량 데이터 수집 시작 (품목: 양파)



전체 날짜 진행:   0%|                                                                 | 1/1461 [00:00<03:34,  6.80it/s]


[20200101] 작업 시작...
totalCnt for 20200101 (page 1-1000): 0

[20200101] 현재 페이지에 데이터가 없으며, 더 이상 가져올 데이터가 없습니다.
[20200101] 수집된 데이터가 없습니다.



[20200102] 작업 시작...


...


[20230505] 리스트 추가 완료 (총 61건)
[20230505] 61건의 데이터 최종 수집 완료.

[20230506] 작업 시작...
totalCnt for 20230506 (page 1-1000): 47
=


전체 날짜 진행:  84%|███████████████████████████████████████████████████▊          | 1222/1461 [16:15<02:58,  1.34it/s]


[20230506] 리스트 추가 완료 (총 47건)
[20230506] 47건의 데이터 최종 수집 완료.



전체 날짜 진행:  84%|███████████████████████████████████████████████████▉          | 1223/1461 [16:15<02:48,  1.41it/s]


[20230507] 작업 시작...
totalCnt for 20230507 (page 1-1000): 0

[20230507] 현재 페이지에 데이터가 없으며, 더 이상 가져올 데이터가 없습니다.
[20230507] 수집된 데이터가 없습니다.
